<a href="https://colab.research.google.com/github/JosephAFerguson/-UserInterface-Proj2/blob/main/DeepLearningJF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json
from datetime import datetime, timedelta

In [118]:
class CryptoEndpoint:
    listingsEndpoint = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
    latestQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/latest"
    historicalQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/historical"

    def __init__(self, apikey) -> None:
        self.headers = {
            'Accepts': 'application/json',
            'X-CMC_PRO_API_KEY': apikey,
        }
        self.coinsInfo = {}
        self.coinsIds = []

    def GetCoinIdentifiers(self):
        session = Session()
        session.headers.update(self.headers)
        response = session.get(
            url=self.listingsEndpoint,
            params={
                "limit": 10,
                "price_min": 1,
                "price_max" : 2
            }
        )

        data = json.loads(response.text)

        for coin in data.get("data", []):
            self.coinsInfo[coin["name"]] = coin["id"]
            self.coinsIds.append(coin["id"])

        print(f"Loaded {len(self.coinsInfo)} coins.")
        return (self.coinsInfo, self.coinsIds)

    def GetCoinLatestPrices(self, coin_id):

        session = Session()
        session.headers.update(self.headers)
        response = session.get(url=self.latestQuotes, params={"id": coin_id})
        data = json.loads(response.text)

        prices = {}
        for coin_id, info in data.get("data", {}).items():
            quote = info["quote"]["USD"]["price"]
            prices[coin_id] = quote

        return prices

    def GetSampleCoinHistoricalData(self, coin_id, days=4):
        session = Session()
        session.headers.update(self.headers)

        end_time = datetime.utcnow()
        start_time = end_time - timedelta(days=days)

        prices = {}

        params = {
            "id": coin_id,
            "time_start": start_time.isoformat(),
            "time_end": end_time.isoformat(),
            "interval": "24h",
        }

        response = session.get(url=self.historicalQuotes, params=params)
        data = json.loads(response.text)
        print(data["status"]["error_code"])
        coin_data = data.get("data", {})

        historicals = []

        for quote in coin_data["quotes"]:
            date = quote.get("timestamp") or quote.get("time_open")
            price = quote["quote"]["USD"]["price"]
            volume = quote["quote"]["USD"]["volume_24h"]
            historicals.append([price,volume])

        returnData = {coin_id: historicals}
        return returnData

In [119]:
ce = CryptoEndpoint(input("Enter API-KEY"))
(coinsInfo, coinsIds) = ce.GetCoinIdentifiers()
data = []
for coinId in coinsIds:
  coinSample = ce.GetSampleCoinHistoricalData(coinId)
  if len(list(coinSample.values())[0]) < 1:
    continue
  data.append(coinSample)
print(coinsInfo)
print(data)

Enter API-KEYa76bd6fc-2b66-4a25-843f-321def3437bd
Loaded 10 coins.
0


/tmp/ipython-input-613701782.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


0
0
0
0
0
0
0
0
0
{'USDC': 3408, 'Toncoin': 11419, 'NEAR Protocol': 6535, 'Worldcoin': 13502, 'Arbitrum': 11841, 'Render': 5690, 'PancakeSwap': 7186, 'Optimism': 11840, 'Lido DAO': 8000, 'Helium': 5665}
[{3408: [[0.9998701269218246, 22210021650.05], [0.999829689087668, 31338846146.03], [0.9996763920156517, 18427357255.67], [1.0000639257225117, 18411640160.46]]}, {11419: [[2.032531418772275, 254647708.88], [1.8407193287024521, 264560589.67], [1.9413092241395933, 144309438.33], [1.96211116732236, 126477596.68]]}, {6535: [[1.9235774486934611, 275058323.99], [1.7495520257235362, 348696694.58], [1.9256104824148519, 210396323.88], [2.092342345321356, 443093080.35]]}, {13502: [[0.7315320588998512, 250197524.8], [0.6738064621870917, 262461882.76], [0.7278177121650083, 142469423.91], [0.7100479092239086, 126747070.04]]}, {11841: [[0.2664608941612087, 295637188.66], [0.243976864145551, 361172957.28], [0.26598616705000566, 194824829.25], [0.2664087778055509, 198954768.87]]}, {5690: [[2.0292134716